In [ ]:
#very promessing and importent, here we take the sliding window approch which aggregate the results of predictions
#of different consecutive "windows" within the conversation. this method allow us to consider longer conversations and thus
#consider the supporter messages (note in almost all other files we didnt do it, we consider only the seeker messages)
#this method learn how to aggregate correctly! that is we train an LSTM model to aggregate predictions
#this model was never deployed but preformed better then most on some importent matrics, consider this idea if you read it!
import pandas as pd
import numpy as np
from tqdm import tqdm_pandas
from tqdm.notebook import tqdm
from transformers import BertModel, BertTokenizerFast, Trainer, TrainingArguments, AutoModelForSequenceClassification, AutoTokenizer, BertForSequenceClassification, BertTokenizer
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,  roc_auc_score, fbeta_score
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
import torch.nn as nn
import warnings
warnings.filterwarnings("ignore")



model_name = 'onlplab/alephbert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert_model = AutoModelForSequenceClassification.from_pretrained(model_name , num_labels=2)

checkpoint_path = "model_weights_curriculum_with_both_seeker_helper.pth"

# Load state dict
state_dict = torch.load(checkpoint_path, map_location="cpu")  # or "cuda" if on GPU

# Load weights into model
bert_model.load_state_dict(state_dict)

# Move to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(device)


/home/ronfr/.conda/envs/alephbert_eval/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at onlplab/alephbert-base and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [2]:
conv_info_path = 'trainDatasets/conv_info.csv'
conv_info_df = pd.read_csv(conv_info_path)
conv_info_df['engagement_id'] = conv_info_df['engagement_id'].astype(str)
conv_info_df = conv_info_df.rename(columns={'gsr': 'label'})
train_conv_df, test_conv_df = train_test_split(conv_info_df, test_size=0.2, stratify=conv_info_df['label'],random_state=42)



messages_with_lables_path = 'trainDatasets/combined_riskfree_riskfull_messages_syntatic_fixed.csv'

messages_with_lables_df = pd.read_csv(messages_with_lables_path)

messages_with_lables_df['engagement_id'] = messages_with_lables_df['engagement_id'].astype(str)
messages_with_lables_df = messages_with_lables_df[messages_with_lables_df['anonymized'].notna()]
messages_with_lables_df['name'] = messages_with_lables_df['name'].fillna('-')

#split to train and test base on conversation split
train_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(train_conv_df["engagement_id"])]
test_labled_messages_df = messages_with_lables_df[messages_with_lables_df["engagement_id"].isin(test_conv_df["engagement_id"])]



all_messages_path = 'trainDatasets/messages_anonymized.csv'

all_messages_df = pd.read_csv(all_messages_path)

all_messages_df['engagement_id'] = all_messages_df['engagement_id'].astype(str)
all_messages_df = all_messages_df[all_messages_df['anonymized'].notna()]
all_messages_df['name'] = all_messages_df['name'].fillna('-')

#split to train and test base on conversation split and concat messages
train_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(train_conv_df["engagement_id"])]
train_all_messages_df = train_all_messages_df.merge(train_conv_df , on="engagement_id")
train_all_messages_df = train_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()

test_all_messages_df = all_messages_df[all_messages_df["engagement_id"].isin(test_conv_df["engagement_id"])]
test_all_messages_df = test_all_messages_df.merge(test_conv_df , on="engagement_id")
test_all_messages_df = test_all_messages_df.groupby('engagement_id').agg({'anonymized': '[SEP]'.join, 'label': 'first'}).reset_index()



In [3]:
from datasets import Dataset as DS
def tokenize(batch):
    return tokenizer(batch['anonymized'], padding='max_length', truncation=True, max_length=512)

def make_test_loader(df, tokenize):
    dataset = DS.from_pandas(df)
    dataset = dataset.map(tokenize, batched=True, batch_size=16)
    dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    test_loader = DataLoader(dataset, batch_size=16, shuffle=False)
    return test_loader

labled_messages_test_loader = make_test_loader(test_labled_messages_df, tokenize)

Map: 100%|██████████| 83832/83832 [00:26<00:00, 3124.78 examples/s]


In [19]:

# Model
def sliding_window_inference(
    conversation,
    tokenizer,
    model,
    max_length=512,
    stride=256,
    device='cuda' if torch.cuda.is_available() else 'cpu'
):
    """
    Performs inference on a conversation using sliding window if needed.

    Args:
        conversation (str): The full conversation text.
        tokenizer: HuggingFace tokenizer.
        model: Trained HuggingFace classification model.
        max_length (int): Max tokens per chunk.
        stride (int): Stride for overlapping chunks.
        device (str): 'cuda' or 'cpu'.

    Returns:
        List[Tensor]: Softmax probabilities per chunk (or one if short).
    """
    model.eval()
    model.to(device)

    # Tokenize without truncation to get full length
    tokens = tokenizer(conversation, return_tensors='pt', truncation=False, add_special_tokens=False)
    input_ids = tokens['input_ids'][0]
    total_len = input_ids.shape[0]

    predictions = []

    # Case 1: The whole conversation fits in one chunk
    if total_len <= max_length:
        padded_input_ids = torch.cat([
            input_ids,
            torch.full((max_length - total_len,), tokenizer.pad_token_id)
        ])
        attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()

        with torch.no_grad():
            outputs = model(
                input_ids=padded_input_ids.unsqueeze(0).to(device),
                attention_mask=attention_mask.unsqueeze(0).to(device)
            )
            logits = outputs.logits.squeeze(0).cpu()
            predictions.append(torch.softmax(logits, dim=-1))

    # Case 2: Conversation is longer than max_length → sliding window
    else:
        for start_idx in range(0, total_len, stride):
            end_idx = start_idx + max_length
            chunk_ids = input_ids[start_idx:end_idx]

            # Pad if shorter than max_length
            if chunk_ids.shape[0] < max_length:
                padding_len = max_length - chunk_ids.shape[0]
                chunk_ids = torch.cat([
                    chunk_ids,
                    torch.full((padding_len,), tokenizer.pad_token_id)
                ])

            attention_mask = (chunk_ids != tokenizer.pad_token_id).long()

            with torch.no_grad():
                outputs = model(
                    input_ids=chunk_ids.unsqueeze(0).to(device),
                    attention_mask=attention_mask.unsqueeze(0).to(device)
                )
                logits = outputs.logits.squeeze(0).cpu()
                predictions.append(torch.softmax(logits, dim=-1))

    return predictions

class RiskAggregator(nn.Module):
    def __init__(self, hidden_size=16):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )
    def forward(self, padded_seq, lengths):
        packed = pack_padded_sequence(padded_seq, lengths, batch_first=True, enforce_sorted=False)
        _, (hn, _) = self.lstm(packed)
        return self.classifier(hn[-1]).squeeze(1)

# Dataset
class ConversationDataset(Dataset):
    def __init__(self, df, tokenizer, model, device):
        self.data = []
        self.labels = []
        model.eval()
        for _, row in df.iterrows():
            conv = row["anonymized"]
            predictions = sliding_window_inference(conv, tokenizer, model)
            p1s = [float(x[1]) for x in predictions]
            self.data.append(torch.tensor(p1s))
            self.labels.append(float(row["label"]))
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx], self.labels[idx]

# Collate function
def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(seq) for seq in sequences])
    padded = pad_sequence(sequences, batch_first=True).unsqueeze(-1)
    return padded.to(device), lengths.cpu().long(), torch.tensor(labels).to(device)



In [5]:
# Load data
train_dataset = ConversationDataset(train_all_messages_df, tokenizer, bert_model, device)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)

Token indices sequence length is longer than the specified maximum sequence length for this model (1139 > 512). Running this sequence through the model will result in indexing errors


In [22]:
# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = RiskAggregator().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

# Train
model.train()
for epoch in range(6):
    total_loss = 0
    for padded, lengths, labels in train_loader:
        optimizer.zero_grad()
        preds = model(padded, lengths)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss:.4f}")

Epoch 1: Loss = 680.4817
Epoch 2: Loss = 527.7425
Epoch 3: Loss = 515.1337
Epoch 4: Loss = 507.0023
Epoch 5: Loss = 503.3210
Epoch 6: Loss = 499.5726


In [7]:
def calc_matrics(labels, preds, pred_probs):
    accuracy = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    roc_auc = roc_auc_score(labels, pred_probs)
    f2 = fbeta_score(labels, preds, beta=2)
    
    print(f'Accuracy: {accuracy}')
    print(f'Precision: {precision}')
    print(f'Recall: {recall}')
    print(f'F1: {f1}')
    print(f'ROC-AUC: {roc_auc}')
    print(f'F2: {f2}')

In [23]:
labels = []
preds = []
pred_probs = []

model.eval()
with torch.no_grad():
    for index, row in test_all_messages_df.iterrows():
        conv = row["anonymized"]
        predictions = sliding_window_inference(
            conversation=conv,
            tokenizer=tokenizer,
            model=bert_model
        )
        sucidal_preds = [float(x[1]) for x in predictions]

        # Convert to tensor and run through the model
        seq_tensor = torch.tensor(sucidal_preds, dtype=torch.float32).unsqueeze(0).unsqueeze(-1).to(device)  # (1, seq_len, 1)
        length_tensor = torch.tensor([len(sucidal_preds)]).cpu().long()
        
        if len(seq_tensor[0]) > 1:
            output = model(seq_tensor, length_tensor)  # shape: (1,)
            prob = output.item()
        else:
            prob = seq_tensor[0].item()

        
        labels.append(row["label"])
        preds.append(int(prob > 0.5))
        pred_probs.append(prob)

# Final evaluation
calc_matrics(labels, preds, pred_probs)

Accuracy: 0.909872664130974
Precision: 0.7722772277227723
Recall: 0.7123287671232876
F1: 0.7410926365795725
ROC-AUC: 0.9235592279490416
F2: 0.7235621521335807


In [9]:
bert_model.eval()
labels = []
preds = []
pred_probs = []

for batch in labled_messages_test_loader:
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    label = batch['label'].to(device)

    with torch.no_grad():
        outputs = bert_model(input_ids, attention_mask=attention_mask)

    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predictions = torch.argmax(logits, dim=-1)

    labels.extend(label.cpu().numpy())
    preds.extend(predictions.cpu().numpy())
    pred_probs.extend(probabilities[:, 1].cpu().numpy())

calc_matrics(labels,preds,pred_probs)

Accuracy: 0.9517010210897986
Precision: 0.7764400377714825
Recall: 0.7532348562922249
F1: 0.7646614356291775
ROC-AUC: 0.9727225553318896
F2: 0.7577642613584001


In [13]:
torch.save(model.state_dict(), f"LSTM_sliding_window(2).pth")